In [1]:
import pandas as pd
from collections import Counter

NETID = "cfinley6"

# 1. Load data
summary_df = pd.read_csv(f"{NETID}_project_summary.csv", sep=";")
stats_df = pd.read_csv(f"{NETID}_project_stats.csv")

# Ensure time is datetime for proper sorting
if pd.api.types.is_numeric_dtype(summary_df['time']):
    summary_df['datetime'] = pd.to_datetime(summary_df['time'], unit='s')
else:
    summary_df['datetime'] = pd.to_datetime(summary_df['time'])

# Sort commits chronologically for accurate slicing
summary_df = summary_df.sort_values(by=['project_wocid', 'datetime'])

gap_commits_list = []

# Helper function to classify commit themes based on keywords
def classify_theme(message):
    msg = str(message).lower()
    if any(bot in msg for bot in ['dependabot', 'github actions', 'snyk', 'bot', 'ci']):
        return 'Automated bot contributions'
    elif any(doc in msg for doc in ['doc', 'readme', 'license', 'wiki']):
        return 'Documentation updates'
    elif any(bug in msg for bug in ['fix', 'bug', 'patch', 'resolve', 'issue']):
        return 'Bug fixes'
    elif any(dep in msg for dep in ['bump', 'dependency', 'update', 'upgrade', 'npm', 'pip']):
        return 'Dependency updates'
    elif any(rel in msg for rel in ['release', 'version', 'tag', 'v1.', 'v2.', 'v0.']):
        return 'Release'
    elif any(sec in msg for sec in ['security', 'cve', 'vulnerability']):
        return 'Security patches'
    elif any(feat in msg for feat in ['add', 'new', 'feature', 'create', 'implement', 'support']):
        return 'Feature development'
    else:
        return 'Other'

# Helper function to get top 2 themes formatted with a colon
def get_top_2_themes(df_slice):
    if df_slice.empty:
        return "N/A:N/A"
    
    themes = df_slice['commit message'].apply(classify_theme).tolist()
    counts = Counter(themes)
    top_themes = [item[0] for item in counts.most_common(2)]
    
    if len(top_themes) == 1:
        top_themes.append("Other") # Pad if only one theme exists
        
    return f"{top_themes[0]}:{top_themes[1]}"

# We will store the new columns for the stats file in these lists
before_themes_list = []
after_themes_list = []

for _, stat_row in stats_df.iterrows():
    prj = stat_row['project'] # or 'Project' depending on your column name
    gap_start = stat_row['LongestGapStart']
    gap_end = stat_row['LongestGapEnd']
    
    # Filter commits for this specific project
    prj_df = summary_df[summary_df['project_wocid'] == prj].copy()
    
    if gap_start == "N/A" or pd.isna(gap_start):
        before_themes_list.append("N/A active Project")
        after_themes_list.append("N/A active Project")
        continue

    # 2. Extract PRE-gap commits (last 10 before gap_start)
    pre_gap_df = prj_df[prj_df['datetime'].dt.strftime('%Y-%m') < gap_start].tail(10)
    for _, row in pre_gap_df.iterrows():
        if 'gap_commits_list' not in locals():
            gap_commits_list = []
        gap_commits_list.append({
            'pre/post': 'pre',
            'prj': prj,
            'commit': row['commit_sha1'],
            'author': row['author'],
            'time': row['time'],
            'message': row['commit message']
        })
        
    # 3. Extract POST-gap commits (first 10 after gap_end)
    post_gap_df = prj_df[prj_df['datetime'].dt.strftime('%Y-%m') > gap_end].head(10)
    for _, row in post_gap_df.iterrows():
        if 'gap_commits_list' not in locals():
            gap_commits_list = []
        gap_commits_list.append({
            'pre/post': 'post',
            'prj': prj,
            'commit': row['commit_sha1'],
            'author': row['author'],
            'time': row['time'],
            'message': row['commit message']
        })

    # 4 & 5. Summarize themes
    before_themes_list.append(get_top_2_themes(pre_gap_df))
    after_themes_list.append(get_top_2_themes(post_gap_df))

# Save the gap commits to CSV
gap_out_df = pd.DataFrame(gap_commits_list)
gap_out_df.to_csv(f"{NETID}_project_gap_commits.csv", index=False)
print(f"Saved {len(gap_out_df)} gap commits to {NETID}_project_gap_commits.csv")

# Append new columns to stats_df and save
stats_df['BeforeThemes'] = before_themes_list
stats_df['AfterThemes'] = after_themes_list
stats_df.to_csv(f"{NETID}_project_stats.csv", index=False)
print(f"Updated {NETID}_project_stats.csv with BeforeThemes and AfterThemes")

Saved 149 gap commits to cfinley6_project_gap_commits.csv
Updated cfinley6_project_stats.csv with BeforeThemes and AfterThemes
